# Error Analysis — "Kenapa Instance Tertentu Buruk?" (margin_part @ 350)

Fokus: menjelaskan *kenapa* sebuah instance mendapat skor buruk pada satu/beberapa metrik —
bukan membuktikan efek max input. Tiga lensa berurutan:

1. **Fitur dialog → skor buruk** — adakah ciri dialog (panjang, turn, multi-domain, penolakan,
   perubahan rencana, domain) yang memprediksi skor buruk? (uji Mann-Whitney + lift)
2. **Disagreement antar-metrik** — pisahkan mode kegagalan: *lancar tapi salah* (halusinasi:
   linguistik ok tapi align jeblok) vs *rusak total* (semua metrik rendah).
3. **Sampel terpandu** — instance buruk dikelompokkan (halusinasi / rusak / penolakan / perubahan
   rencana) untuk dibaca & dilabeli sebab manual. Diekspor ke CSV untuk pelabelan.

> Prasyarat: jalankan **setelah** sel-sel Results_Analysis yang mendefinisikan `res` (dengan kolom
> metrik + `prediction` + `reference`), `texts`, `detect_domains`, `n_words`, `DOMAIN_KEYWORDS`,
> `metric_cols`. Ambang "buruk" = persentil 10 pada `align_score` (ubah di sel bila perlu).

## Sel Error Analysis 3-Lensa

In [1]:
# ============================================================
#  ERROR ANALYSIS — "kenapa instance tertentu buruk?" (margin_part @ 350)
#  3 lensa: (1) fitur dialog -> skor buruk  (2) disagreement antar-metrik
#           (3) taksonomi manual terpandu
#  Butuh dari notebook: res (sudah punya kolom metrik + prediction + reference),
#                       texts, detect_domains, n_words, metric_cols
# ============================================================
import pandas as pd, numpy as np, re, textwrap
from scipy.stats import mannwhitneyu

PRED_COL, REF_COL = "prediction", "reference"
LING = ["coherence","relevance","fluency","consistency"]
FOCUS = "align_score"

# --- pastikan fitur dasar ada ---
df = res.copy().reset_index(drop=True)
if "_dialogue" not in df.columns: df["_dialogue"] = pd.Series(texts).reset_index(drop=True).values
if "_nwords"   not in df.columns: df["_nwords"]   = df["_dialogue"].apply(n_words)
if "_domains"  not in df.columns: df["_domains"]  = df["_dialogue"].apply(detect_domains)

# --- fitur turunan dari teks dialog ---
def n_turns(t):  return len([x for x in str(t).split("\n") if x.strip()])
REFUSAL = ["i'm sorry","i am sorry","i cannot","i can't","we don't have","there are no",
           "no restaurants","not able","unable","apologize","unfortunately"]
CHANGE  = ["how about","instead","actually","change","different","never mind","else",
           "what about","on second thought","or maybe"]
df["_nturns"]      = df["_dialogue"].apply(n_turns)
df["_ndomains"]    = df["_domains"].apply(len)
df["_multidomain"] = df["_ndomains"] > 1
df["_refusal"]     = df["_dialogue"].str.lower().apply(lambda t: any(k in t for k in REFUSAL))
df["_change"]      = df["_dialogue"].str.lower().apply(lambda t: any(k in t for k in CHANGE))

# definisi "buruk" pada FOCUS = persentil 10 (eksplisit, reproducible)
thr = df[FOCUS].quantile(0.10)
df["_bad"] = df[FOCUS] <= thr
print(f"Ambang buruk ({FOCUS} <= p10) = {thr:.3f}  -> {df['_bad'].sum()} instance buruk dari {len(df)}")

# ============ LENSA 1: fitur dialog -> skor buruk ============
print("\n" + "="*64)
print("LENSA 1 — fitur dialog yang memprediksi skor buruk")
print("="*64)
num_feats = ["_nwords","_nturns","_ndomains"]
print(f"\n{'fitur':14s}{'buruk(mean)':>14s}{'baik(mean)':>14s}{'p(MWU)':>12s}")
for f in num_feats:
    bad, good = df[df._bad][f], df[~df._bad][f]
    try: _,p = mannwhitneyu(bad, good)
    except ValueError: p=np.nan
    flag=" *" if p<0.05 else ""
    print(f"{f:14s}{bad.mean():14.2f}{good.mean():14.2f}{p:12.3g}{flag}")

print("\nFitur biner (rate pada buruk vs baik):")
for f in ["_multidomain","_refusal","_change"]:
    rb, rg = df[df._bad][f].mean(), df[~df._bad][f].mean()
    lift = rb/rg if rg>0 else float('inf')
    print(f"  {f:14s} buruk={rb:.2%}  baik={rg:.2%}  (lift {lift:.2f}x)")

# domain over-represented di kelompok buruk
def share(sub):
    c={}
    for ds in sub["_domains"]:
        for d in ds: c[d]=c.get(d,0)+1
    return {d: c.get(d,0)/(len(sub) or 1) for d in DOMAIN_KEYWORDS}
sb, sg = share(df[df._bad]), share(df[~df._bad])
print("\nDomain (rate buruk vs baik, lift):")
for d in DOMAIN_KEYWORDS:
    lift = sb[d]/sg[d] if sg[d]>0 else float('inf')
    print(f"  {d:11s} buruk={sb[d]:.2%}  baik={sg[d]:.2%}  (lift {lift:.2f}x)")

# ============ LENSA 2: disagreement antar-metrik ============
print("\n" + "="*64)
print("LENSA 2 — pola disagreement antar-metrik")
print("="*64)
z = (df[LING+[FOCUS]] - df[LING+[FOCUS]].mean()) / df[LING+[FOCUS]].std(ddof=0)
df["_ling_z"]  = z[LING].mean(axis=1)
df["_focus_z"] = z[FOCUS]
df["_gap"]     = df["_ling_z"] - df["_focus_z"]   # tinggi = linguistik bagus tapi align jelek
print("Korelasi antar-metrik (Spearman):")
print(df[LING+[FOCUS]].corr(method="spearman").round(2).to_string())
print(f"""
Dua mode kegagalan:
  'lancar tapi salah' (gap tinggi): fluency/coherence ok, {FOCUS} jeblok -> kandidat HALUSINASI
  'rusak total'       (semua z<0) : semua metrik rendah bareng -> output degenerate / off-topic""")
df["_mode"] = np.where(df["_gap"] > 1.0, "lancar_tapi_salah",
              np.where((z[LING].mean(axis=1) < -0.5) & (z[FOCUS] < -0.5), "rusak_total", "lainnya"))
print("\nDistribusi mode pada instance BURUK:")
print(df[df._bad]["_mode"].value_counts().to_string())

# ============ LENSA 3: sampel terpandu utk taksonomi manual ============
def show(sub, judul, k=4):
    print("\n"+"="*78); print(judul); print("="*78)
    for i,r in sub.head(k).iterrows():
        tags=[t for t,v in [("refusal",r._refusal),("change",r._change),("multidom",r._multidomain)] if v]
        print(f"\n[row {i}] {int(r._nwords)}w {int(r._nturns)}turn {r._domains} {tags}")
        print(f"   align={r[FOCUS]:.3f} | "+" ".join(f"{m[:3]}={r[m]:.2f}" for m in LING))
        print("   DIALOG:", textwrap.shorten(str(r['_dialogue']).replace(chr(10),' / '), 230))
        print("   PRED  :", textwrap.shorten(str(r[PRED_COL]), 200))
        print("   REF   :", textwrap.shorten(str(r[REF_COL]), 200))

print("\n" + "="*64); print("LENSA 3 — sampel terpandu (baca & beri label sebab)"); print("="*64)
show(df[df._bad & (df._mode=="lancar_tapi_salah")].sort_values("_gap",ascending=False),
     "A) LANCAR TAPI SALAH — kandidat halusinasi (linguistik ok, align jeblok)")
show(df[df._bad & (df._mode=="rusak_total")].sort_values(FOCUS),
     "B) RUSAK TOTAL — semua metrik rendah")
show(df[df._bad & df._refusal].sort_values(FOCUS),
     "C) BURUK + ada PENOLAKAN dalam dialog")
show(df[df._bad & df._change].sort_values(FOCUS),
     "D) BURUK + ada PERUBAHAN RENCANA dalam dialog")

# ekspor instance buruk + fitur utk pelabelan manual
out_cols=["_nwords","_nturns","_ndomains","_multidomain","_refusal","_change","_mode"]+LING+[FOCUS,PRED_COL,REF_COL]
df.loc[df._bad, out_cols].to_csv("bad_instances_for_labeling.csv", index_label="row")
print("\n>> Disimpan: bad_instances_for_labeling.csv  (instance buruk + fitur, siap dilabeli manual)")


NameError: name 'res' is not defined